# Modèle lourd : DistilBERT mesuré par CodeCarbon (Kaggle) · Ons · Séance 3

Données (code commun) → DistilBERT, 1 époque → accuracy + `emissions_lourd.csv`.
Notebook **autonome** : le code du dépôt est recopié ci-dessous, aucun jeton GitHub n'est nécessaire.

**Avant *Run All*** (panneau de droite) :
1. *Session options → Accelerator* : **GPU T4 x2**.
2. *Internet* : **On**.

Déroulé (≈ 15 min au total) : test court (500 exemples), puis **3 vrais runs** `lourd_run1..3`
(10 000 / 2 000). Toutes les sorties vont dans `resultats/kaggle/`, au format lu par les scripts de Ghayth.

In [ ]:
!nvidia-smi

Un seul GPU calcule (`CUDA_VISIBLE_DEVICES=0`) et CodeCarbon ne mesure que lui (`gpu_ids=[0]`). `CUDA_DEVICE_ORDER=PCI_BUS_ID` aligne la numérotation CUDA sur celle de CodeCarbon (NVML).

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.makedirs("src", exist_ok=True)
open("src/__init__.py", "w").close()

In [ ]:
!pip install -q codecarbon

## 1. Code du dépôt (copie de `src/commun.py` et `src/modele_lourd.py`)

In [ ]:
%%writefile src/commun.py
"""Code commun : identique pour tous (mêmes données, même mesure).

Copié tel quel de la fiche de répartition (séance 2). Ne pas modifier seul :
toute modification se décide en équipe, sinon les mesures ne sont plus comparables.
"""

from datasets import load_dataset
from codecarbon import EmissionsTracker

SEED = 42


def load_data(n_train=10_000, n_test=2_000, seed=SEED):
    # "ag_news" seul est refusé par huggingface_hub récent (id « namespace/name » exigé) :
    # fancyzhx/ag_news est le même jeu de données sous son identifiant actuel.
    ds = load_dataset("fancyzhx/ag_news")
    train = ds["train"].shuffle(seed=seed).select(range(n_train))
    test = ds["test"].shuffle(seed=seed).select(range(n_test))
    return train, test  # colonnes : "text", "label" (4 classes)


def mesurer(nom, fonction_entrainement, **options_tracker):
    # Séance 3 : mesure chaque seconde (comme Ali et Ghayth, entraînements courts) ;
    # options_tracker permet en plus de choisir le fichier de sortie et les GPU mesurés
    # (ex. gpu_ids=[0] sur Kaggle, où CodeCarbon compte sinon le 2e T4 inutilisé).
    options = {"project_name": nom, "output_file": "emissions.csv", "measure_power_secs": 1}
    options.update(options_tracker)
    tracker = EmissionsTracker(**options)
    tracker.start()
    accuracy = fonction_entrainement()
    kg = tracker.stop()
    print(f"{nom} | accuracy={accuracy:.4f} | {kg*1000:.3f} g CO2eq")
    return accuracy, kg


# Ali : mesurer("leger", train_light)    Ons : mesurer("lourd", train_heavy)

In [ ]:
%%writefile src/modele_lourd.py
"""Modèle lourd : fine-tuning de DistilBERT, 1 époque (responsable : Ons).

Chaîne complète : données (code commun) -> DistilBERT (HF Trainer) -> accuracy
-> emissions.csv (CodeCarbon, via mesurer()).

À lancer sur Kaggle avec un GPU T4 (notebooks/lourd_distilbert_kaggle.ipynb) :

    python -m src.modele_lourd --test-court     # 500 exemples : valide la mesure
    python -m src.modele_lourd --runs 3         # vrais runs lourd_run1..3 : 10 000 / 2 000

Sorties dans resultats/kaggle/ (format lu par src/reproductibilite.py et src/sensibilite.py) :
- emissions_lourd.csv : une ligne CodeCarbon par run (project_name = lourd_test_court, lourd_run1…) ;
- <nom>.json : accuracy, durée, GPU, hyperparamètres, estimation de durée du vrai run.
"""

from __future__ import annotations

import argparse
import json
import platform
import time
from pathlib import Path
from typing import Any

from src.commun import SEED, load_data, mesurer

# Hyperparamètres du modèle lourd (à reporter dans docs/modele_lourd.md).
HEAVY_PARAMS: dict[str, Any] = {
    "model_name": "distilbert-base-uncased",
    "num_labels": 4,
    "epochs": 1,
    "batch_size": 16,
    "learning_rate": 5e-5,
    "weight_decay": 0.01,
    "max_length": 128,
}

N_TRAIN_VRAI_RUN = 10_000
LIMITE_MINUTES = 20
DOSSIER_SORTIE = "resultats/kaggle"
FICHIER_EMISSIONS = "emissions_lourd.csv"


def infos_materiel() -> dict[str, Any]:
    """Décrit le matériel utilisé (pour le document « Hypothèses »)."""
    import torch

    infos: dict[str, Any] = {
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "cpu": platform.processor() or platform.machine(),
        "torch": torch.__version__,
    }
    if torch.cuda.is_available():
        infos["gpu"] = torch.cuda.get_device_name(0)
    return infos


def train_heavy(
    train: Any,
    test: Any,
    n_train: int | None = None,
    n_test: int | None = None,
    params: dict[str, Any] | None = None,
    nom: str = "lourd",
    dossier_resultats: str = DOSSIER_SORTIE,
) -> float:
    """Fine-tune DistilBERT sur `train`, renvoie l'accuracy sur `test`.

    `train` et `test` sont les datasets renvoyés par load_data() (colonnes text, label).
    n_train / n_test limitent le nombre d'exemples (test court).
    """
    import torch
    import transformers
    from sklearn.metrics import accuracy_score
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        DataCollatorWithPadding,
        Trainer,
        TrainingArguments,
        set_seed,
    )

    p = {**HEAVY_PARAMS, **(params or {})}
    set_seed(SEED)

    if n_train is not None:
        train = train.select(range(min(n_train, len(train))))
    if n_test is not None:
        test = test.select(range(min(n_test, len(test))))

    tokenizer = AutoTokenizer.from_pretrained(p["model_name"])

    def encoder(batch: dict[str, list]) -> dict[str, list]:
        return tokenizer(batch["text"], truncation=True, max_length=p["max_length"])

    ds_train = train.map(encoder, batched=True, remove_columns=["text"])
    ds_test = test.map(encoder, batched=True, remove_columns=["text"])

    model = AutoModelForSequenceClassification.from_pretrained(
        p["model_name"], num_labels=p["num_labels"]
    )
    args = TrainingArguments(
        output_dir=f"outputs/{nom}",
        num_train_epochs=p["epochs"],
        per_device_train_batch_size=p["batch_size"],
        per_device_eval_batch_size=p["batch_size"] * 2,
        learning_rate=p["learning_rate"],
        weight_decay=p["weight_decay"],
        seed=SEED,
        fp16=torch.cuda.is_available(),
        save_strategy="no",
        report_to="none",
        logging_steps=50,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_train,
        data_collator=DataCollatorWithPadding(tokenizer),
    )

    debut = time.perf_counter()
    trainer.train()
    duree_train = time.perf_counter() - debut

    preds = trainer.predict(ds_test).predictions.argmax(axis=-1)
    accuracy = float(accuracy_score(ds_test["label"], preds))

    # Estimation linéaire de la durée d'entraînement du vrai run (10 000 exemples).
    estimation_min = duree_train / len(ds_train) * N_TRAIN_VRAI_RUN * p["epochs"] / 60
    infos = {
        "nom": nom,
        "accuracy": accuracy,
        "n_train": len(ds_train),
        "n_test": len(ds_test),
        "duree_entrainement_s": round(duree_train, 1),
        "estimation_vrai_run_min": round(estimation_min, 1),
        "respecte_limite_20_min": estimation_min < LIMITE_MINUTES,
        "hyperparametres": p,
        "materiel": infos_materiel(),
        "transformers": transformers.__version__,
    }
    Path(dossier_resultats).mkdir(exist_ok=True)
    chemin = Path(dossier_resultats) / f"{nom}.json"
    chemin.write_text(json.dumps(infos, indent=2, ensure_ascii=False), encoding="utf-8")
    print(json.dumps(infos, indent=2, ensure_ascii=False))
    return accuracy


def main() -> None:
    parser = argparse.ArgumentParser(description="Modèle lourd mesuré par CodeCarbon")
    parser.add_argument(
        "--test-court", action="store_true", help="500 exemples train / 200 test"
    )
    parser.add_argument(
        "--runs", type=int, default=1, help="nombre de vrais runs (lourd_run1, lourd_run2…)"
    )
    parser.add_argument("--premier", type=int, default=1, help="numéro du premier run")
    args = parser.parse_args()

    import torch

    Path(DOSSIER_SORTIE).mkdir(parents=True, exist_ok=True)
    options_tracker: dict[str, Any] = {
        "output_dir": DOSSIER_SORTIE,
        "output_file": FICHIER_EMISSIONS,
    }
    if torch.cuda.is_available():
        # Ne mesurer que le GPU qui calcule (Kaggle en fournit deux).
        options_tracker["gpu_ids"] = [0]

    train, test = load_data()
    if args.test_court:
        taches = [("lourd_test_court", 500, 200)]
    else:
        taches = [
            (f"lourd_run{i}", None, None) for i in range(args.premier, args.premier + args.runs)
        ]

    for nom, n_train, n_test in taches:
        mesurer(
            nom,
            lambda: train_heavy(train, test, n_train, n_test, nom=nom),
            **options_tracker,
        )


if __name__ == "__main__":
    main()

## 2. Test court (500 / 200) : vérifier la mesure et la durée estimée

In [ ]:
!python -m src.modele_lourd --test-court

In [ ]:
import json
res = json.load(open("resultats/kaggle/lourd_test_court.json"))
print("GPU :", res["materiel"].get("gpu"), "| accuracy :", res["accuracy"],
      "| vrai run estimé :", res["estimation_vrai_run_min"], "min | < 20 min :", res["respecte_limite_20_min"])

## 3. Trois vrais runs (10 000 / 2 000) : `lourd_run1`, `lourd_run2`, `lourd_run3`

In [ ]:
!python -m src.modele_lourd --runs 3

## 4. Récapitulatif (moyenne ± écart-type sur les 3 runs)

In [ ]:
import glob
import pandas as pd

em = pd.read_csv("resultats/kaggle/emissions_lourd.csv")
print("GPU mesurés par CodeCarbon :", em["gpu_count"].unique(), em["gpu_model"].unique())
runs = em[em["project_name"].str.startswith("lourd_run")].copy()
accs = {json.load(open(f))["nom"]: json.load(open(f)) for f in glob.glob("resultats/kaggle/lourd_run*.json")}
runs["accuracy"] = runs["project_name"].map(lambda n: accs[n]["accuracy"])
runs["entrainement_s"] = runs["project_name"].map(lambda n: accs[n]["duree_entrainement_s"])
runs["energie_Wh"] = runs["energy_consumed"] * 1000
runs["emissions_g"] = runs["emissions"] * 1000
cols = ["accuracy", "entrainement_s", "duration", "energie_Wh", "emissions_g", "gpu_power", "ram_power", "cpu_power"]
display(runs[["project_name"] + cols])
recap = runs[cols].agg(["mean", "std"]).T
recap["cv_%"] = 100 * recap["std"] / recap["mean"]
display(recap.round(4))
print("Intensité implicite (g CO2/kWh) :", round((runs["emissions"] / runs["energy_consumed"]).mean() * 1000))

## 5. Récupérer les résultats

Télécharger le dossier `resultats/kaggle/` (cellule suivante, puis onglet *Output* → `resultats_kaggle.zip`),
le décompresser dans le dépôt à la place de `resultats/kaggle/`, puis committer et pousser.

In [ ]:
!cd resultats && zip -qr ../resultats_kaggle.zip kaggle && ls -l ../resultats_kaggle.zip